# **Smoke Detector (Fire Alarm Detector)**
----

## **Data Definition** 

| Feature | Description |
|-------|-------------|
| UTC (datetime) | Time stamp the sensor records. |
| Temperature[C] (float) | Environment temperature (Celsius). |
| Humidity[%] (float) | Percentage of environment humidity. |
| TVOC[ppb] (int) | Total Volatile Organic Compounds (parts per billion). |
| eCO2[ppm] (int) | CO₂ concentration (parts per million). |
| Raw H2 (int) | Number of H₂ gas molecules. |
| Raw Ethanol (int) | Number of Ethanol gas molecules. |
| Pressure[hPa] (float) | Air pressure (hectopascal, 1 hPa = 100 Pa). |
| PM1.0 (float) | Number of particles < 1.0 micrometer. |
| PM2.5 (float) | Number of particles between 1.0 and 2.5 micrometer. |
| NC0.5 (float) | Particle concentration < 0.5 micrometer. |
| NC1.0 (float) | Particle concentration between 0.5 and 1.0 micrometer. |
| NC2.5 (float) | Particle concentration between 1.0 and 2.5 micrometer. |
| CNT (int) | Sample counter. |

----

**Output Value: data to be predicted**

| Feature | Description |
|-------|-------------|
| Fire Alarm (binary) | 0 = No fire detected, 1 = Fire detected. |

## **Import Libraries**

In [90]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split

# define project root
from pathlib import Path
project_root = Path().resolve().parent

# set project root to the most priority of sys path
import sys
sys.path.insert(0, str(project_root))

from src.utils import *

## **Load Config**

In [91]:
# load config
config = load_config()
config

{'column_datetime': 'UTC',
 'columns_float': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'pm25',
  'nc05',
  'nc10',
  'nc25'],
 'columns_int': ['tvoc', 'co2', 'raw_h2', 'raw_ethanol', 'fire_alarm'],
 'features': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'tvoc',
  'co2',
  'raw_h2',
  'raw_ethanol'],
 'label': 'fire_alarm',
 'random_state': 123,
 'range_co2': [400, 60000],
 'range_humidity_pct': [10.74, 75.2],
 'range_pm10': [0.0, 14333.69],
 'range_pressure': [930.852, 939.861],
 'range_raw_ethanol': [15317, 21410],
 'range_raw_h2': [10668, 13803],
 'range_temperature': [-22.01, 59.93],
 'range_tvoc': [0, 60000],
 'raw_data': 'data/raw/smoke_detection_iot.csv',
 'size_train_valid': 0.2,
 'size_valid_test': 0.5}

## **Load Data**

In [92]:
def load_data(path_data: str) -> pd.DataFrame:
    """
    Load csv file and return it as the pandas dataframe.

    Parameter:
    ----------
    path_data : str
        Loaded data path.
    
    Return:
    ------
    data : pd.DataFrame
        Loaded dataset
    """
    # load csv file
    data = pd.read_csv(path_data)

    # check the original data shape
    print(f'Original data shape: {data.shape}')

    # drop any duplicate data
    total_duplicates = data.duplicated().sum()
    print(f'Total duplicates detected: {total_duplicates}')

    data = data.drop_duplicates(keep='last')
    print(f'Dropped data shape: {data.shape}')

    return data

In [93]:
# load the raw dataset
raw_dataset = load_data(path_data=project_root/config['raw_data'])

Original data shape: (62630, 15)
Total duplicates detected: 0
Dropped data shape: (62630, 15)


In [94]:
# sanity check data head
raw_dataset.head()

,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


## **Data Validation**

In [95]:
# check the number of missing values
raw_dataset.isna().sum()

UTC               0
Temperature[C]    0
Humidity[%]       0
TVOC[ppb]         0
eCO2[ppm]         0
Raw H2            0
Raw Ethanol       0
Pressure[hPa]     0
PM1.0             0
PM2.5             0
NC0.5             0
NC1.0             0
NC2.5             0
CNT               0
Fire Alarm        0
dtype: int64

In [96]:
# check data types of each feature
raw_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UTC             62630 non-null  int64  
 1   Temperature[C]  62630 non-null  float64
 2   Humidity[%]     62630 non-null  float64
 3   TVOC[ppb]       62630 non-null  int64  
 4   eCO2[ppm]       62630 non-null  int64  
 5   Raw H2          62630 non-null  int64  
 6   Raw Ethanol     62630 non-null  int64  
 7   Pressure[hPa]   62630 non-null  float64
 8   PM1.0           62630 non-null  float64
 9   PM2.5           62630 non-null  float64
 10  NC0.5           62630 non-null  float64
 11  NC1.0           62630 non-null  float64
 12  NC2.5           62630 non-null  float64
 13  CNT             62630 non-null  int64  
 14  Fire Alarm      62630 non-null  int64  
dtypes: float64(8), int64(7)
memory usage: 7.2 MB


UTC should be in datetime data type

In [97]:
# convert the utc column into datetime type
raw_dataset[config['column_datetime']] = pd.to_datetime(raw_dataset[config['column_datetime']], unit="s")
raw_dataset.head()

,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,2022-06-09 00:08:51,20.000,57.36,0,400,12306,18520,939.735,0.0,0.0,0.0,0.0,0.0,0,0
1,2022-06-09 00:08:52,20.015,56.67,0,400,12345,18651,939.744,0.0,0.0,0.0,0.0,0.0,1,0
2,2022-06-09 00:08:53,20.029,55.96,0,400,12374,18764,939.738,0.0,0.0,0.0,0.0,0.0,2,0
3,2022-06-09 00:08:54,20.044,55.28,0,400,12390,18849,939.736,0.0,0.0,0.0,0.0,0.0,3,0
4,2022-06-09 00:08:55,20.059,54.69,0,400,12403,18921,939.744,0.0,0.0,0.0,0.0,0.0,4,0


In [98]:
# drop the CNT column
raw_dataset = raw_dataset.drop(columns=['CNT'])

In [99]:
# rename the column
new_column = ['utc', 'temperature', 'humidity_pct', 'tvoc', 'co2', 'raw_h2', 'raw_ethanol', 'pressure', 'pm10', 'pm25', 'nc05', 'nc10', 'nc25', 'fire_alarm']

raw_dataset.columns = new_column

In [100]:
# recheck the data info
raw_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62630 entries, 0 to 62629
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   utc           62630 non-null  datetime64[ns]
 1   temperature   62630 non-null  float64       
 2   humidity_pct  62630 non-null  float64       
 3   tvoc          62630 non-null  int64         
 4   co2           62630 non-null  int64         
 5   raw_h2        62630 non-null  int64         
 6   raw_ethanol   62630 non-null  int64         
 7   pressure      62630 non-null  float64       
 8   pm10          62630 non-null  float64       
 9   pm25          62630 non-null  float64       
 10  nc05          62630 non-null  float64       
 11  nc10          62630 non-null  float64       
 12  nc25          62630 non-null  float64       
 13  fire_alarm    62630 non-null  int64         
dtypes: datetime64[ns](1), float64(8), int64(5)
memory usage: 6.7 MB


In [101]:
# serialize the validated data
path_data_validated = project_root/'data/interim/validated_data.pkl'
serialize_data(data=raw_dataset, path=path_data_validated)

Data serialized to /home/bagaskoroah/smoke_detector/data/interim/validated_data.pkl


['/home/bagaskoroah/smoke_detector/data/interim/validated_data.pkl']

In [102]:
# update the configuration file
config = update_config(
    key='path_data_validated',
    value='data/interim/path_data_validated.pkl',
    config=config
)

Config has been successfully updated. 
Key: path_data_validated 
Value: data/interim/path_data_validated.pkl



## **Update Range of Data**

In [103]:
# update the range of numerical features
cols = config['features']
cols_float = config['columns_float']

print(f'Selected features: {cols}\n')

param_keys = [f'range_{col}' for col in cols]

for col, key in zip(cols, param_keys):
    if col in cols_float:
        min_value = float(np.min(raw_dataset[col]))
        max_value = float(np.max(raw_dataset[col]))
    else:
        min_value = int(np.min(raw_dataset[col]))
        max_value = int(np.max(raw_dataset[col]))

    value = [min_value, max_value]

    config = update_config(
        key=key,
        value=value,
        config=config
    )

Selected features: ['temperature', 'humidity_pct', 'pressure', 'pm10', 'tvoc', 'co2', 'raw_h2', 'raw_ethanol']

Config has been successfully updated. 
Key: range_temperature 
Value: [-22.01, 59.93]

Config has been successfully updated. 
Key: range_humidity_pct 
Value: [10.74, 75.2]

Config has been successfully updated. 
Key: range_pressure 
Value: [930.852, 939.861]

Config has been successfully updated. 
Key: range_pm10 
Value: [0.0, 14333.69]

Config has been successfully updated. 
Key: range_tvoc 
Value: [0, 60000]

Config has been successfully updated. 
Key: range_co2 
Value: [400, 60000]

Config has been successfully updated. 
Key: range_raw_h2 
Value: [10668, 13803]

Config has been successfully updated. 
Key: range_raw_ethanol 
Value: [15317, 21410]



## **Data Defense**

In [104]:
# define a data defense function
def data_defense(data, config):
    """
    Do data defense to check the data types and range.

    Parameters:
    ----------
    data : pd.DataFrame
        Validated data
    config : dict
        Loaded configuration file
    
    Returns:
    -------
    None (void function)
    """
    # generate total data entries
    n_data = len(data)

    # list of columns
    cols_float = config['columns_float']
    cols_int = config['columns_int']

    # check data types
    assert data.select_dtypes('float').columns.tolist() == cols_float, "an error occurs in float columns"
    assert data.select_dtypes('int').columns.tolist() == cols_int, "an error occurs in integer columns"

    # check range values of data
    for col in config['features']:
        min_value = config[f'range_{col}'][0]
        max_value = config[f'range_{col}'][1]
        assert data[col].between(min_value, max_value).sum() == n_data, f'an error occurs in {col} range'

In [105]:
# run the data defense process
data_defense(data=raw_dataset, config=config)

## **Data Split**

In [106]:
# define an input-output split function
def split_input_output(data, config):
    """
    Split the input(X) and output (y).

    Parameters:
    ----------
    data : pd.DataFrame
        The processed dataset.

    config : dict
        Loaded configuration parameters.

    Returns:
    -------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.    
    """

    # ensure raw data immutable
    data = data.copy()

    # split x and y
    X = data[config['features']]
    y = data[config['label']]

    print(f'Original data shape     : {data.shape}')
    print(f'Selected features: {config['features']}')
    print(f'X data shape            : {X.shape}')
    print(f'y data shape            : {y.shape}')

    return X, y

In [107]:
# split input output
X, y = split_input_output(
    data=raw_dataset,
    config=config
)

Original data shape     : (62630, 14)
Selected features: ['temperature', 'humidity_pct', 'pressure', 'pm10', 'tvoc', 'co2', 'raw_h2', 'raw_ethanol']
X data shape            : (62630, 8)
y data shape            : (62630,)


In [108]:
# define a train-test split function
def split_train_test(X, y, test_size, random_state=config['random_state']):
    """
    Split the train and test set.

    Parameters:
    ----------
    X : pd.DataFrame
        The input data.

    y : pd.Series
        The output data.

    test_size : float
        The proportion of test set.

    random_state : int, default = 123
        For reproducibility

    Returns:
    -------
    X_train, X_test : pd.DataFrame
        The train and test input.

    y_train, y_test : pd.Series
        The train and test output.
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    print(f"X_train shape : {X_train.shape}")
    print(f"y_train shape : {y_train.shape}")
    print(f"X_test shape  : {X_test.shape}")
    print(f"y_test shape  : {y_test.shape}\n")

    return X_train, X_test, y_train, y_test

In [109]:
# train vs not train
X_train, X_not_train, y_train, y_not_train = split_train_test(
    X=X,
    y=y,
    test_size=config['size_train_valid']
)

X_valid, X_test, y_valid, y_test = split_train_test(
    X=X_not_train,
    y=y_not_train,
    test_size=config['size_valid_test']
)


X_train shape : (50104, 8)
y_train shape : (50104,)
X_test shape  : (12526, 8)
y_test shape  : (12526,)

X_train shape : (6263, 8)
y_train shape : (6263,)
X_test shape  : (6263, 8)
y_test shape  : (6263,)



## **Data Serialization**

In [110]:
# serialize all the splitted data

data_config = {
    'X_train': X_train,
    'X_valid': X_valid,
    'X_test': X_test,
    'y_train': y_train,
    'y_valid': y_valid,
    'y_test': y_test
}

for key, value in data_config.items():
    serialize_data(data=value, path=project_root/f'data/interim/{key}.pkl')

    config = update_config(
        key=f'path_data_{key}',
        value=f'data/interim/{key}.pkl',
        config=config
    )

Data serialized to /home/bagaskoroah/smoke_detector/data/interim/X_train.pkl
Config has been successfully updated. 
Key: path_data_X_train 
Value: data/interim/X_train.pkl

Data serialized to /home/bagaskoroah/smoke_detector/data/interim/X_valid.pkl


Config has been successfully updated. 
Key: path_data_X_valid 
Value: data/interim/X_valid.pkl

Data serialized to /home/bagaskoroah/smoke_detector/data/interim/X_test.pkl
Config has been successfully updated. 
Key: path_data_X_test 
Value: data/interim/X_test.pkl

Data serialized to /home/bagaskoroah/smoke_detector/data/interim/y_train.pkl
Config has been successfully updated. 
Key: path_data_y_train 
Value: data/interim/y_train.pkl

Data serialized to /home/bagaskoroah/smoke_detector/data/interim/y_valid.pkl
Config has been successfully updated. 
Key: path_data_y_valid 
Value: data/interim/y_valid.pkl

Data serialized to /home/bagaskoroah/smoke_detector/data/interim/y_test.pkl
Config has been successfully updated. 
Key: path_data_y_test 
Value: data/interim/y_test.pkl



In [111]:
# sanity check config file and its updates
config

{'column_datetime': 'UTC',
 'columns_float': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'pm25',
  'nc05',
  'nc10',
  'nc25'],
 'columns_int': ['tvoc', 'co2', 'raw_h2', 'raw_ethanol', 'fire_alarm'],
 'features': ['temperature',
  'humidity_pct',
  'pressure',
  'pm10',
  'tvoc',
  'co2',
  'raw_h2',
  'raw_ethanol'],
 'label': 'fire_alarm',
 'path_data_X_test': 'data/interim/X_test.pkl',
 'path_data_X_train': 'data/interim/X_train.pkl',
 'path_data_X_valid': 'data/interim/X_valid.pkl',
 'path_data_validated': 'data/interim/path_data_validated.pkl',
 'path_data_y_test': 'data/interim/y_test.pkl',
 'path_data_y_train': 'data/interim/y_train.pkl',
 'path_data_y_valid': 'data/interim/y_valid.pkl',
 'random_state': 123,
 'range_co2': [400, 60000],
 'range_humidity_pct': [10.74, 75.2],
 'range_pm10': [0.0, 14333.69],
 'range_pressure': [930.852, 939.861],
 'range_raw_ethanol': [15317, 21410],
 'range_raw_h2': [10668, 13803],
 'range_temperature': [-22.01, 59.93],
 'range_tvo